In [1]:
!pip install pdfplumber camelot-py tabula-py pandas

   ---------------------------------------- 0.0/12.0 MB ? eta -:--:--
   ----------- ---------------------------- 3.4/12.0 MB 25.5 MB/s eta 0:00:01
   ------------------------- -------------- 7.6/12.0 MB 21.5 MB/s eta 0:00:01
   --------------------------------- ------ 10.0/12.0 MB 18.1 MB/s eta 0:00:01
   ---------------------------------------- 12.0/12.0 MB 16.1 MB/s  0:00:00



[notice] A new release of pip is available: 26.0.1 -> 26.1.2
[notice] To update, run: python.exe -m pip install --upgrade pip


In [39]:
import pdfplumber

pdf_path = "D:/sudhendra/learning projects/GraphRAG/data/Apple23 report.pdf"

with pdfplumber.open(pdf_path) as pdf:
    page = pdf.pages[24]
    tables = page.extract_tables()
    print(tables)

[[['iPhone (1) $ 200,583 (2) % $ 205,489 7 % $ 191,973'], ['Mac (1) 29,357 (27) % 40,177 14 % 35,190'], ['iPad (1) 28,300 (3) % 29,292 (8) % 31,862'], ['Wearables, Home and Accessories (1) 39,845 (3) % 41,241 7 % 38,367'], ['Services (2) 85,200 9 % 78,129 14 % 68,425'], ['Total net sales $ 383,285 (3) % $ 394,328 8 % $ 365,817']]]


In [41]:
import pdfplumber
import pandas as pd

pdf_path = "D:/sudhendra/learning projects/GraphRAG/data/Apple23 report.pdf"

table_rows = []

with pdfplumber.open(pdf_path) as pdf:
    for page_index, page in enumerate(pdf.pages):
        tables = page.extract_tables()

        if not tables:
            continue

        for table_index, table in enumerate(tables):
            if not table or len(table) < 2:
                continue

            header = table[0]
            header_text = " | ".join([str(cell) for cell in header if cell])

            for row_index, row in enumerate(table[1:], start=1):
                row_text = " | ".join([str(cell) for cell in row if cell])

                if row_text.strip():
                    combined_text = f"Table headers: {header_text}\nTable row: {row_text}"

                    table_rows.append({
                        "page": page_index + 1,
                        "table_id": f"page_{page_index+1}_table_{table_index}_row_{row_index}",
                        "text": combined_text
                    })

tables_df = pd.DataFrame(table_rows)
tables_df.head()

,page,table_id,text
0,3,page_3_table_0_row_1,Table headers: Item 1. | Business | 1\nTable r...
1,3,page_3_table_0_row_2,Table headers: Item 1. | Business | 1\nTable r...
2,3,page_3_table_0_row_3,Table headers: Item 1. | Business | 1\nTable r...
3,3,page_3_table_0_row_4,Table headers: Item 1. | Business | 1\nTable r...
4,3,page_3_table_0_row_5,Table headers: Item 1. | Business | 1\nTable r...


In [42]:
tables_df[tables_df["text"].str.contains("net sales", case=False, na=False)]

,page,table_id,text
25,24,page_24_table_0_row_5,"Table headers: Americas $ 162,560 (4) % $ 169,..."
30,25,page_25_table_0_row_5,"Table headers: iPhone (1) $ 200,583 (2) % $ 20..."
35,26,page_26_table_2_row_1,Table headers: Percentage of total net sales 8...
36,26,page_26_table_2_row_2,Table headers: Percentage of total net sales 8...
37,26,page_26_table_2_row_3,Table headers: Percentage of total net sales 8...
38,26,page_26_table_2_row_4,Table headers: Percentage of total net sales 8...
46,31,page_31_table_0_row_2,"Table headers: Products $ 298,085 $ 316,199 $ ..."
169,38,page_38_table_0_row_4,"Table headers: Mac (1) 29,357 40,177 35,190\nT..."
315,50,page_50_table_0_row_1,"Table headers: Net sales $ 162,560 $ 169,658 $..."
316,50,page_50_table_0_row_3,"Table headers: Net sales $ 162,560 $ 169,658 $..."


In [43]:
import chromadb
from sentence_transformers import SentenceTransformer

embedding_model = SentenceTransformer("all-MiniLM-L6-v2")

chroma_client = chromadb.PersistentClient(path="D:/sudhendra/learning projects/GraphRAG/vector_db")

collection = chroma_client.get_or_create_collection(
    name="apple_2023_10k"
)

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 8823.62it/s]


In [44]:
table_docs = tables_df["text"].astype(str).tolist()
table_ids = tables_df["table_id"].astype(str).tolist()

table_metadatas = [
    {
        "page": int(row["page"]),
        "type": "table_row",
        "table_id": str(row["table_id"])
    }
    for _, row in tables_df.iterrows()
]


In [45]:
table_embeddings = embedding_model.encode(
    table_docs,
    batch_size=32,
    show_progress_bar=True
).tolist()

Batches: 100%|██████████| 15/15 [00:01<00:00, 11.31it/s]


In [47]:
collection.upsert( #collection.upsert can also be used it works smartly, it checks for duplicates id and then updates and add new records
    ids=table_ids,
    documents=table_docs,
    embeddings=table_embeddings,
    metadatas=table_metadatas
)

In [48]:
query = "Apple total net sales 2023 2022 2021"

query_embedding = embedding_model.encode(query).tolist()

results = collection.query(
    query_embeddings=[query_embedding],
    n_results=10
)

for i in range(len(results["documents"][0])):
    print("Result:", i + 1)
    print("Page:", results["metadatas"][0][i]["page"])
    print("Type:", results["metadatas"][0][i].get("type", "text_chunk"))
    print("Distance:", results["distances"][0][i])
    print(results["documents"][0][i])
    print("-" * 100)

Result: 1
Page: 25
Type: table_row
Distance: 0.82890385389328
Table headers: iPhone (1) $ 200,583 (2) % $ 205,489 7 % $ 191,973
Table row: Total net sales $ 383,285 (3) % $ 394,328 8 % $ 365,817
----------------------------------------------------------------------------------------------------
Result: 2
Page: 25
Type: table_row
Distance: 0.8576056957244873
iPhone (1) $ 200,583 (2) % $ 205,489 7 % $ 191,973
----------------------------------------------------------------------------------------------------
Result: 3
Page: 38
Type: table_row
Distance: 0.8986167311668396
Table headers: Mac (1) 29,357 40,177 35,190
Table row: Total net sales $ 383,285 $ 394,328 $ 365,817
----------------------------------------------------------------------------------------------------
Result: 4
Page: 25
Type: table_row
Distance: 1.0016112327575684
Table headers: iPhone (1) $ 200,583 (2) % $ 205,489 7 % $ 191,973
Table row: Mac (1) 29,357 (27) % 40,177 14 % 35,190
----------------------------------------

In [49]:
def make_table_row_text(row):
    clean_row = [str(cell).strip() for cell in row if cell and str(cell).strip()]
    
    row_text = " | ".join(clean_row)

    return f"""
Financial table row.
Columns: 2023 | 2022 | 2021
Row: {row_text}
"""

In [50]:
table_rows = []

with pdfplumber.open(pdf_path) as pdf:
    for page_index, page in enumerate(pdf.pages):
        tables = page.extract_tables()

        if not tables:
            continue

        for table_index, table in enumerate(tables):
            if not table:
                continue

            for row_index, row in enumerate(table):
                row_text = make_table_row_text(row)

                if row_text.strip():
                    table_rows.append({
                        "page": page_index + 1,
                        "table_id": f"page_{page_index+1}_table_{table_index}_row_{row_index}",
                        "text": row_text
                    })

tables_df = pd.DataFrame(table_rows)

In [51]:
import chromadb
from sentence_transformers import SentenceTransformer

embedding_model = SentenceTransformer("all-MiniLM-L6-v2")

chroma_client = chromadb.PersistentClient(path="D:/sudhendra/learning projects/GraphRAG/vector_db")

collection = chroma_client.get_or_create_collection(
    name="apple_2023_10k"
)

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 9236.38it/s]


In [52]:
table_docs = tables_df["text"].astype(str).tolist()
table_ids = tables_df["table_id"].astype(str).tolist()

table_metadatas = [
    {
        "page": int(row["page"]),
        "type": "table_row",
        "table_id": str(row["table_id"])
    }
    for _, row in tables_df.iterrows()
]


In [55]:
collection.upsert( #collection.upsert can also be used it works smartly, it checks for duplicates id and then updates and add new records
    ids=table_ids,
    documents=table_docs,
    embeddings=table_embeddings,
    metadatas=table_metadatas
)

ValueError: Unequal lengths for fields: ids: 585, metadatas: 585, embeddings: 455, documents: 585 in upsert.

In [56]:
print(len(table_ids))
print(len(table_docs))
print(len(table_metadatas))
print(len(table_embeddings))

585
585
585
455


In [57]:
table_embeddings = embedding_model.encode(
    table_docs,
    batch_size=32,
    show_progress_bar=True
).tolist()

Batches: 100%|██████████| 19/19 [00:01<00:00, 13.75it/s]


In [58]:
print(len(table_ids))
print(len(table_docs))
print(len(table_metadatas))
print(len(table_embeddings))

585
585
585
585


In [59]:
collection.upsert(
    ids=table_ids,
    documents=table_docs,
    embeddings=table_embeddings,
    metadatas=table_metadatas
)

In [60]:
query = "Apple total net sales 2023 2022 2021"

query_embedding = embedding_model.encode(query).tolist()

results = collection.query(
    query_embeddings=[query_embedding],
    n_results=10
)

for i in range(len(results["documents"][0])):
    print("Result:", i + 1)
    print("Page:", results["metadatas"][0][i]["page"])
    print("Type:", results["metadatas"][0][i].get("type", "text_chunk"))
    print("Distance:", results["distances"][0][i])
    print(results["documents"][0][i])
    print("-" * 100)

Result: 1
Page: 76
Type: table_row
Distance: 0.7274990081787109

Financial table row.
Columns: 2023 | 2022 | 2021
Row: Apple Sales International Limited Ireland

----------------------------------------------------------------------------------------------------
Result: 2
Page: 76
Type: table_row
Distance: 0.7784092426300049

Financial table row.
Columns: 2023 | 2022 | 2021
Row: Apple Operations International Limited Ireland

----------------------------------------------------------------------------------------------------
Result: 3
Page: 76
Type: table_row
Distance: 0.786385715007782

Financial table row.
Columns: 2023 | 2022 | 2021
Row: Apple Distribution International Limited Ireland

----------------------------------------------------------------------------------------------------
Result: 4
Page: 76
Type: table_row
Distance: 0.812629759311676

Financial table row.
Columns: 2023 | 2022 | 2021
Row: Apple Canada Inc. Canada

--------------------------------------------------------

In [61]:
important_keywords = [
    "net sales",
    "total net sales",
    "revenue",
    "gross margin",
    "operating income",
    "net income",
    "cash",
    "assets",
    "liabilities",
    "research and development",
    "selling, general and administrative"
]

filtered_tables_df = tables_df[
    tables_df["text"].str.contains("|".join(important_keywords), case=False, na=False)
].copy()

len(filtered_tables_df)

97

In [62]:
table_docs = tables_df["text"].astype(str).tolist()
table_ids = tables_df["table_id"].astype(str).tolist()

table_metadatas = [
    {
        "page": int(row["page"]),
        "type": "table_row",
        "table_id": str(row["table_id"])
    }
    for _, row in tables_df.iterrows()
]

table_embeddings = embedding_model.encode(
    table_docs,
    batch_size=32,
    show_progress_bar=True
).tolist()

collection.upsert(
    ids=table_ids,
    documents=table_docs,
    embeddings=table_embeddings,
    metadatas=table_metadatas
)

Batches: 100%|██████████| 19/19 [00:01<00:00, 14.09it/s]


In [71]:
query = "Total net sales 383,285 394,328 365,817 Apple 2023 2022 2021"

query_embedding = embedding_model.encode(query).tolist()

results = collection.query(
    query_embeddings=[query_embedding],
    n_results=10,
)

for i in range(len(results["documents"][0])):
    doc = results["documents"][0][i]
    
    if "net sales" not in doc.lower():
        continue
        
    print("Result:", i + 1)
    print("Page:", results["metadatas"][0][i]["page"])
    print("Distance:", results["distances"][0][i])
    print(doc)
    print("-" * 100)

Result: 1
Page: 38
Distance: 0.7993164658546448

Financial table row.
Columns: 2023 | 2022 | 2021
Row: Total net sales $ 383,285 $ 394,328 $ 365,817

----------------------------------------------------------------------------------------------------
Result: 2
Page: 51
Distance: 0.7993164658546448

Financial table row.
Columns: 2023 | 2022 | 2021
Row: Total net sales $ 383,285 $ 394,328 $ 365,817

----------------------------------------------------------------------------------------------------
Result: 3
Page: 24
Distance: 0.8003140687942505

Financial table row.
Columns: 2023 | 2022 | 2021
Row: Total net sales $ 383,285 (3) % $ 394,328 8 % $ 365,817

----------------------------------------------------------------------------------------------------
Result: 4
Page: 25
Distance: 0.8003140687942505

Financial table row.
Columns: 2023 | 2022 | 2021
Row: Total net sales $ 383,285 (3) % $ 394,328 8 % $ 365,817

-----------------------------------------------------------------------------

Hybrid Retrieval

In [78]:
!pip install langchain_text_splitters


[notice] A new release of pip is available: 26.0.1 -> 26.1.2
[notice] To update, run: python.exe -m pip install --upgrade pip


In [80]:
import pandas as pd
from pathlib import Path
from langchain_text_splitters import RecursiveCharacterTextSplitter

# Load extracted text CSV
csv_path = Path("../data/apple23_extractedtxt.csv")

df = pd.read_csv(csv_path)

# Remove empty pages
df = df.dropna(subset=["text"])
df = df[df["text"].str.strip() != ""]
df.reset_index(drop=True, inplace=True)

# Create page-wise documents
documents = []

for _, row in df.iterrows():
    documents.append({
        "page": int(row["page"]),
        "text": row["text"]
    })

# Chunking
splitter = RecursiveCharacterTextSplitter(
    chunk_size=1500,
    chunk_overlap=250,
    separators=["\n\n", "\n", ".", " ", ""]
)

chunks = []

for doc in documents:
    page_chunks = splitter.split_text(doc["text"])

    for i, chunk in enumerate(page_chunks):
        chunks.append({
            "page": doc["page"],
            "chunk_id": f"text_page_{doc['page']}_chunk_{i}",
            "type": "text_chunk",
            "text": chunk
        })

# Convert to DataFrame
chunks_df = pd.DataFrame(chunks)

# Check sample
chunks_df.head()

,page,chunk_id,type,text
0,1,text_page_1_chunk_0,text_chunk,UNITED STATES\nSECURITIES AND EXCHANGE COMMISS...
1,1,text_page_1_chunk_1,text_chunk,—\nThe Nasdaq Stock Market LLC\n1.375% Notes d...
2,2,text_page_2_chunk_0,text_chunk,Indicate by check mark whether the Registrant ...
3,2,text_page_2_chunk_1,text_chunk,any new or revised financial accounting standa...
4,2,text_page_2_chunk_2,text_chunk,day of the Registrant’s most recently complete...


In [81]:
text_docs = chunks_df["text"].astype(str).tolist()

text_ids = chunks_df["chunk_id"].astype(str).tolist()

text_metadatas = [
    {
        "page": int(row["page"]),
        "type": "text_chunk",
        "chunk_id": str(row["chunk_id"])
    }
    for _, row in chunks_df.iterrows()
]

In [82]:
text_embeddings = embedding_model.encode(
    text_docs,
    batch_size=32,
    show_progress_bar=True
).tolist()

Batches: 100%|██████████| 8/8 [00:04<00:00,  1.92it/s]


In [83]:
collection.upsert(
    ids=text_ids,
    documents=text_docs,
    embeddings=text_embeddings,
    metadatas=text_metadatas
)

In [112]:
def hybrid_retrieval(query, n_text = 5,n_tables = 5):
    query_embedding =  embedding_model.encode(query).tolist()


    text_results = collection.query(
        query_embeddings = [query_embedding],
        n_results = n_text,
        where = {"type" : "text_chunk"}
    )

    table_results = collection.query(
        query_embeddings = [query_embedding],
        n_results = n_tables,
        where = {"type" : "table_row"}   
    )

    retrieved = []

    for i in range(len(table_results["documents"][0])):
        retrieved.append({
            "source_type" : "table_row",
            "text" : table_results["documents"][0][i],
            "page" : table_results["metadatas"][0][i]["page"],
            "distance" : table_results["distances"][0][i]
        })

    for i in range(len(text_results["documents"][0])):
        retrieved.append({
            "source_type" : "text_chunk",
            "text" : table_results["documents"][0][i],
            "page" : table_results["metadatas"][0][i]["page"],
            "distance" : table_results["distances"][0][i]
        })

    return retrieved




In [113]:
def build_hybrid_context(chunks):
    context = ""

    for i, chunk in enumerate(chunks, start=1):
        context += f"\n[Source {i} | Page {chunk['page']} | Type: {chunk['source_type']}]\n"
        context += chunk["text"]
        context += "\n"

    return context

In [114]:
import ollama

In [119]:
def generate_hybrid_answer(query, n_text=5, n_tables=5):
    chunks = hybrid_retrieval(query, n_text=n_text, n_tables=n_tables)
    context = build_hybrid_context(chunks)

    prompt = f"""
You are a financial report assistant.

Answer using ONLY the provided Apple 2023 Form 10-K context.
If you see a table row named "Total net sales", use it as the primary source for total net sales.
Do not use iPhone, Mac, or Services rows as substitutes for total net sales.

Rules:
1. Use table_row sources as primary evidence for numerical financial questions.
2. Use text_chunk sources for explanation and context.
3. Do not invent missing numbers.
4. Perform calculations only using numbers explicitly present in the context.
5. Show calculation steps.
6. Give page references.
7. If the context is insufficient, say so clearly.

Context:
{context}

Question:
{query}

Answer:
"""

    response = ollama.chat(
        model="mistral",
        messages=[{"role": "user", "content": prompt}],
        options={"temperature": 0}
    )

    return response["message"]["content"]

In [120]:
print(generate_hybrid_answer(
    "Compare Apple's 2023 and 2022 total net sales. Show difference and percentage change.",
    n_text=5,
    n_tables=5
))

 To compare Apple's 2023 and 2022 total net sales, we need to find the corresponding numbers from the "Total net sales" table row. However, the provided context does not contain such a table row. Therefore, I cannot provide the exact numerical comparison or percentage change as required.

To calculate the total net sales for each year, we can sum up the percentages of total net sales from the given rows (Source 6, Source 7, and Source 8) since they add up to 100%.

For 2023:
Total net sales = (Percentage of total net sales for 2023 * Total net sales in 2023)
                        = (14% + 8% + 7%) * Total net sales in 2023 (unknown)

For 2022:
Total net sales = (Percentage of total net sales for 2022 * Total net sales in 2022)
                        = (13% + 7% + 6%) * Total net sales in 2022 (unknown)

To find the difference and percentage change, we would need to know the total net sales for both years. Unfortunately, without this information, I cannot provide the comparison or pe

fixing the retrieval process , because earlier it was not calculating

In [117]:
chunks = hybrid_retrieval(
    "Total net sales 383,285 394,328 365,817 Apple 2023 2022 2021",
    n_text=3,
    n_tables=10
)

for i, chunk in enumerate(chunks, start=1):
    print("SOURCE", i)
    print("TYPE:", chunk["source_type"])
    print("PAGE:", chunk["page"])
    print("DISTANCE:", chunk["distance"])
    print(chunk["text"][:1000])
    print("-" * 100)

SOURCE 1
TYPE: table_row
PAGE: 38
DISTANCE: 0.7993164658546448

Financial table row.
Columns: 2023 | 2022 | 2021
Row: Total net sales $ 383,285 $ 394,328 $ 365,817

----------------------------------------------------------------------------------------------------
SOURCE 2
TYPE: table_row
PAGE: 51
DISTANCE: 0.7993164658546448

Financial table row.
Columns: 2023 | 2022 | 2021
Row: Total net sales $ 383,285 $ 394,328 $ 365,817

----------------------------------------------------------------------------------------------------
SOURCE 3
TYPE: table_row
PAGE: 24
DISTANCE: 0.8003140687942505

Financial table row.
Columns: 2023 | 2022 | 2021
Row: Total net sales $ 383,285 (3) % $ 394,328 8 % $ 365,817

----------------------------------------------------------------------------------------------------
SOURCE 4
TYPE: table_row
PAGE: 25
DISTANCE: 0.8003140687942505

Financial table row.
Columns: 2023 | 2022 | 2021
Row: Total net sales $ 383,285 (3) % $ 394,328 8 % $ 365,817

-----------------

In [118]:
def rewrite_financial_query(query):
    query_lower = query.lower()

    if "total net sales" in query_lower or "net sales" in query_lower:
        return "Total net sales 383,285 394,328 365,817 Apple 2023 2022 2021"

    return query

In [121]:
def hybrid_retrieval(query, n_text=5, n_tables=5):
    retrieval_query = rewrite_financial_query(query)
    query_embedding = embedding_model.encode(retrieval_query).tolist()

    text_results = collection.query(
        query_embeddings=[query_embedding],
        n_results=n_text,
        where={"type": "text_chunk"}
    )

    table_results = collection.query(
        query_embeddings=[query_embedding],
        n_results=n_tables,
        where={"type": "table_row"}
    )

    retrieved = []

    for i in range(len(table_results["documents"][0])):
        retrieved.append({
            "source_type": "table_row",
            "text": table_results["documents"][0][i],
            "page": table_results["metadatas"][0][i]["page"],
            "distance": table_results["distances"][0][i]
        })

    for i in range(len(text_results["documents"][0])):
        retrieved.append({
            "source_type": "text_chunk",
            "text": text_results["documents"][0][i],
            "page": text_results["metadatas"][0][i]["page"],
            "distance": text_results["distances"][0][i]
        })

    return retrieved

In [122]:
def build_hybrid_context(chunks):
    context = ""

    for i, chunk in enumerate(chunks, start=1):
        context += f"\n[Source {i} | Page {chunk['page']} | Type: {chunk['source_type']}]\n"
        context += chunk["text"]
        context += "\n"

    return context

In [127]:
def generate_hybrid_answer(query, n_text=5, n_tables=5):
    chunks = hybrid_retrieval(query, n_text=n_text, n_tables=n_tables)
    context = build_hybrid_context(chunks)

    prompt = f"""
You are a financial report assistant.

Answer using ONLY the provided Apple 2023 Form 10-K context.

Rules:
1. Use table_row sources as primary evidence for numerical financial questions.
2. Use text_chunk sources only for explanation/context.
3. Do not invent missing numbers.
4. Perform calculations only using numbers explicitly present in the context.
5. If you see "Total net sales", use that row for total net sales.
6. Give a clean, structured financial-analysis answer.
7. Always mention values are in millions unless the context says otherwise.
8. Always include source references.

Context:
{context}

Question:
{query}

Answer in this exact format:

### Answer
[One clear sentence answering the question.]

### Values Used
- 2023 total net sales: $___ million
- 2022 total net sales: $___ million

### Calculation
Difference = 2023 value - 2022 value  
Difference = $___ - $___ = $___ million

Percentage change = Difference / 2022 value × 100  
Percentage change = ___%

### Interpretation
[Explain whether sales increased or decreased and what it means.]

### Sources
- [Source number | Page number | Type]
"""

    response = ollama.chat(
        model="mistral",
        messages=[{"role": "user", "content": prompt}],
        options={"temperature": 0}
    )

    return response["message"]["content"]

In [128]:
print(generate_hybrid_answer(
    "Compare Apple's 2023 and 2022 total net sales. Show difference and percentage change.",
    n_text=5,
    n_tables=5
))

 ### Answer
Apple's 2023 total net sales were lower than in 2022.

### Values Used
- 2023 total net sales: $383,285 million
- 2022 total net sales: $394,328 million

### Calculation
Difference = 2023 value - 2022 value
Difference = $383,285 - $394,328 = $-11,043 million

Percentage change = Difference / 2022 value × 100
Percentage change = (-$11,043) / $394,328 × 100 = -2.8%

### Interpretation
Apple's total net sales decreased by 2.8% in 2023 compared to 2022, indicating a slight decline in overall revenue.

### Sources
- [Source 1 | Page 38 | Type: table_row]
- [Source 2 | Page 51 | Type: table_row]
